In [ ]:
import re
import sys
import csv
import numpy as np
from pathlib import Path as ph
import matplotlib.pyplot as plt

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
def process_visualize_file_components(combined):
    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    attentions = ['mha', 'moh', 'gqa', 'swh', 'gla', 'rfa']
    networks = ['mlp', 'moe', 'lor', 'swi', 'gln', 'ggl'] 
    n_layers = [4,8,16]

    # Extract header and data
    header = statistic[0]
    header[0] = header[0].lstrip('\ufeff').lstrip('\ufeff') # Remove BOM if present
    data = statistic[1:]

    # Find columns index
    x_col_a = header.index("baby's brain")
    w_col_b = header.index("c_device")
    w_col_c = header.index("n_layers")
    w_col_d = header.index("c_attention")
    w_col_e = header.index("c_network")
    w_col_f = header.index("inference_quality_execution")
    w_col_g = header.index("total_time_execution")  # New column for time

    # Collect all time values first for consistent normalization
    all_times_list = []
    layer_data = {}
    
    for n_layer in n_layers:
        combos_quality = {}
        combos_time = {}
        combos_finetuned_quality = {}
        combos_finetuned_time = {}
        
        for row in data:
            if str(row[w_col_b]) == 'gpu' and int(row[w_col_c]) == n_layer:
                att = row[w_col_d]
                net = row[w_col_e]
                quality = float(row[w_col_f].replace("/100", ""))
                time_val = float(row[w_col_g]) if row[w_col_g] else np.nan
                
                if not str(row[x_col_a]).endswith('_finetuned'):
                    combos_quality[(att, net)] = quality
                    combos_time[(att, net)] = time_val
                    all_times_list.append(time_val)
                else:
                    combos_finetuned_quality[(att, net)] = quality
                    combos_finetuned_time[(att, net)] = time_val
                    all_times_list.append(time_val)
        
        layer_data[n_layer] = {
            'combos_quality': combos_quality,
            'combos_time': combos_time,
            'combos_finetuned_quality': combos_finetuned_quality,
            'combos_finetuned_time': combos_finetuned_time
        }
    
    time_min = np.nanmin(all_times_list)
    time_max = np.nanmax(all_times_list)

    def plot_quad_triangles(ax, baseline_quality, finetuned_quality, baseline_time, finetuned_time,
                            title, time_min, time_max):
        """
        Plot cells divided into 4 triangles meeting at center:
        - Left triangle: Baseline Quality (Green)
        - Top triangle: Finetuned Quality (Green)
        - Right triangle: Baseline Time (Red)
        - Bottom triangle: Finetuned Time (Red)
        """
        n_rows, n_cols = baseline_quality.shape

        # Create colormaps for each triangle
        cmap_base_quality = plt.cm.Greens
        cmap_fine_quality = plt.cm.Greens
        cmap_base_time = plt.cm.Reds
        cmap_fine_time = plt.cm.Reds

        # Normalize quality (0-100) and time
        norm_quality = plt.Normalize(0, 100)
        norm_time = plt.Normalize(time_min, time_max)

        for i in range(n_rows):
            for j in range(n_cols):
                bq_val = baseline_quality[i, j]
                fq_val = finetuned_quality[i, j]
                bt_val = baseline_time[i, j]
                ft_val = finetuned_time[i, j]

                # Cell corners and center
                x0, x1 = j - 0.5, j + 0.5  # Left, Right
                y0, y1 = i - 0.5, i + 0.5  # Top, Bottom
                cx, cy = j, i              # Center

                # Left triangle: Baseline Quality (Green)
                if not np.isnan(bq_val):
                    color = cmap_base_quality(norm_quality(bq_val))
                    triangle = plt.Polygon([(x0, y1), (x0, y0), (cx, cy)],
                                          facecolor=color, edgecolor='white', linewidth=0.5)
                    ax.add_patch(triangle)

                # Top triangle: Finetuned Quality (Green)
                if not np.isnan(fq_val):
                    color = cmap_fine_quality(norm_quality(fq_val))
                    triangle = plt.Polygon([(x0, y0), (x1, y0), (cx, cy)],
                                          facecolor=color, edgecolor='white', linewidth=0.5)
                    ax.add_patch(triangle)

                # Right triangle: Baseline Time (Red)
                if not np.isnan(bt_val):
                    color = cmap_base_time(norm_time(bt_val))
                    triangle = plt.Polygon([(x1, y0), (x1, y1), (cx, cy)],
                                          facecolor=color, edgecolor='white', linewidth=0.5)
                    ax.add_patch(triangle)

                # Bottom triangle: Finetuned Time (Red)
                if not np.isnan(ft_val):
                    color = cmap_fine_time(norm_time(ft_val))
                    triangle = plt.Polygon([(x1, y1), (x0, y1), (cx, cy)],
                                          facecolor=color, edgecolor='white', linewidth=0.5)
                    ax.add_patch(triangle)
        
        ax.set_xlim(-0.5, n_cols - 0.5)
        ax.set_ylim(n_rows - 0.5, -0.5)
        ax.set_xticks(range(len(attentions)))
        ax.set_xticklabels(attentions, rotation=45)
        ax.set_yticks(range(len(networks)))
        ax.set_yticklabels(networks)
        ax.set_xlabel('Attention Mechanism')
        ax.set_ylabel('Network Mechanism')
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        ax.set_aspect('equal')

    def build_matrices(combos_quality, combos_time, combos_finetuned_quality, combos_finetuned_time):
        matrix_quality = []
        matrix_time = []
        matrix_finetuned_quality = []
        matrix_finetuned_time = []
        
        for net in networks:
            row_quality = []
            row_time = []
            row_finetuned_quality = []
            row_finetuned_time = []
            for att in attentions:
                row_quality.append(combos_quality.get((att, net), np.nan))
                row_time.append(combos_time.get((att, net), np.nan))
                row_finetuned_quality.append(combos_finetuned_quality.get((att, net), np.nan))
                row_finetuned_time.append(combos_finetuned_time.get((att, net), np.nan))
            matrix_quality.append(row_quality)
            matrix_time.append(row_time)
            matrix_finetuned_quality.append(row_finetuned_quality)
            matrix_finetuned_time.append(row_finetuned_time)
        
        return (np.array(matrix_quality), np.array(matrix_time), 
                np.array(matrix_finetuned_quality), np.array(matrix_finetuned_time))

    def add_quad_colorbars(fig, time_min, time_max):
        # Quality colorbar (Green) - baseline (left) & finetuned (top) share same range
        cbar_ax1 = fig.add_axes([0.25, 0.06, 0.22, 0.02])
        sm1 = plt.cm.ScalarMappable(cmap=plt.cm.Greens, norm=plt.Normalize(0, 100))
        sm1.set_array([])
        cbar1 = fig.colorbar(sm1, cax=cbar_ax1, orientation='horizontal')
        cbar1.set_label('Quality: Baseline (left) / Finetuned (top)', fontsize=10)

        # Time colorbar (Red) - baseline (right) & finetuned (bottom) share same range
        cbar_ax2 = fig.add_axes([0.55, 0.06, 0.22, 0.02])
        sm2 = plt.cm.ScalarMappable(cmap=plt.cm.Reds, norm=plt.Normalize(time_min, time_max))
        sm2.set_array([])
        cbar2 = fig.colorbar(sm2, cax=cbar_ax2, orientation='horizontal')
        cbar2.set_label('Time: Baseline (right) / Finetuned (bottom)', fontsize=10)

    if combined:
        # All layer figures in same plot
        fig, axes = plt.subplots(1, len(n_layers), figsize=(8 * len(n_layers), 10))
        if len(n_layers) == 1:
            axes = [axes]

        for idx, n_layer in enumerate(n_layers):
            ld = layer_data[n_layer]
            matrix_quality, matrix_time, matrix_finetuned_quality, matrix_finetuned_time = build_matrices(
                ld['combos_quality'], ld['combos_time'],
                ld['combos_finetuned_quality'], ld['combos_finetuned_time']
            )

            ax = axes[idx]
            plot_quad_triangles(ax, matrix_quality, matrix_finetuned_quality, 
                               matrix_time, matrix_finetuned_time,
                               f'Combinations for n_layers={n_layer}', time_min, time_max)

        plt.subplots_adjust(left=0.06, right=0.96, bottom=0.14, top=0.92, wspace=0.2)
        add_quad_colorbars(fig, time_min, time_max)
        plt.savefig(f"{statistics_path}/statistic_experiments_matrixes.png", bbox_inches='tight')
        plt.show()
    else:
        # Each layer in its own plot
        for n_layer in n_layers:
            ld = layer_data[n_layer]
            matrix_quality, matrix_time, matrix_finetuned_quality, matrix_finetuned_time = build_matrices(
                ld['combos_quality'], ld['combos_time'],
                ld['combos_finetuned_quality'], ld['combos_finetuned_time']
            )

            fig, ax = plt.subplots(figsize=(12, 10))
            plot_quad_triangles(ax, matrix_quality, matrix_finetuned_quality,
                               matrix_time, matrix_finetuned_time,
                               f'Combinations for n_layers={n_layer}', time_min, time_max)
            add_quad_colorbars(fig, time_min, time_max)
            plt.show()

In [ ]:
process_visualize_file_components(True)